# Processing Larsim

## Disaggregate the 3h accumulated variables

In [ ]:
def diff_within_blocks_vectorized(da, block_size=3):
    """
    Vectorized version using groupby.
    """
    # Create block labels for grouping
    time_index = np.arange(len(da.time))
    block_labels = np.ceil(time_index / 3).astype(int)
    
    # Add block labels as a coordinate
    da_with_blocks = da.assign_coords(block=('time', block_labels))
    
    # Define a function to apply to each block
    def block_diff(block):
        if len(block.time) > 1:
            # Create a mask for all positions except the first in each block
            time_pos = xr.DataArray(range(len(block.time)), dims=['time'], coords={'time': block.time})
            mask = time_pos > 0
            
            # Shift the block by one position along time
            shifted = block.shift(time=1)
            
            # Use where to apply the difference only where mask is True
            return block.where(~mask, block - shifted)
        return block
    
    # Apply the function to each block and concatenate results
    result = da_with_blocks.groupby('block').map(block_diff)
    
    # Remove the block coordinate
    result = result.drop_vars('block')
    
    return result

In [ ]:
path_data = "/automount/agh/s6tifohr/july21_eval/data/LARSIM/"
exp_name = "blcklst_sat"
ds = xr.open_mfdataset([f"{path_data}/{exp_name}/daily_files/fc_R03B08_N02_sel.202107{dd:02}.nc" for dd in range(6,17)])

In [ ]:
# Disaggregate grid scale rain
da_diff_rain_gsp = diff_within_blocks_vectorized(ds["RAIN_GSP"])
ds["RAIN_GSP"] = da_diff_rain_gsp

# Disaggregate total precipitation
da_diff_tot_prec = diff_within_blocks_vectorized(ds["TOT_PREC"])
ds["TOT_PREC"] = da_diff_tot_prec

In [ ]:
# Output daily files:
for dd in range(6,17):
    time_slice = slice(np.datetime64(f"2021-07-{dd:02}T00"), np.datetime64(f"2021-07-{dd:02}T23"))
    path_out = f"{path_data}/{exp_name}/disaggregated/larsim_{exp_name}_det_202107{dd:02}.nc"
    ds.sel(time=time_slice).to_netcdf(path_out)